# Random Forest: The Wisdom of the Crowd

**Chapter 5: Advanced Classification Methods**

## Learning Objectives

- Understand ensemble learning and bagging
- Build a Random Forest classifier
- Interpret feature importances
- Compare Random Forest with single decision trees
- Tune Random Forest hyperparameters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## The Intuition: Ensemble Learning

**Problem:** A single decision tree is unstable - small changes in data lead to completely different trees.

**Solution:** Build many trees and let them vote!

**Random Forest = Bagging + Feature Randomness**

1. **Bagging (Bootstrap Aggregating)**: Each tree trains on a random sample of data (with replacement)
2. **Feature Randomness**: Each split considers only a random subset of features

These techniques ensure **diverse trees** that make different mistakes. The majority vote cancels out individual errors.

## Load Data: Match Outcome Prediction

We'll predict match outcomes (Win/Draw/Loss) based on team statistics.

In [ ]:
# For demonstration, let's create a synthetic match dataset
# In practice, you'd aggregate event data to match level

np.random.seed(42)
n_matches = 200

# Simulate match statistics
match_data = pd.DataFrame({
    'home_shots': np.random.poisson(12, n_matches),
    'away_shots': np.random.poisson(10, n_matches),
    'home_passes': np.random.normal(450, 80, n_matches),
    'away_passes': np.random.normal(420, 75, n_matches),
    'home_tackles': np.random.poisson(15, n_matches),
    'away_tackles': np.random.poisson(16, n_matches)
})

# Create outcome based on statistics (with some randomness)
def determine_outcome(row):
    home_strength = row['home_shots'] * 0.5 + row['home_passes'] * 0.001 - row['away_tackles'] * 0.1
    away_strength = row['away_shots'] * 0.5 + row['away_passes'] * 0.001 - row['home_tackles'] * 0.1
    diff = home_strength - away_strength + np.random.normal(0, 2)
    if diff > 1:
        return 'Home Win'
    elif diff < -1:
        return 'Away Win'
    else:
        return 'Draw'

match_data['outcome'] = match_data.apply(determine_outcome, axis=1)

print(f"Total matches: {len(match_data)}")
print(f"\nOutcome distribution:")
print(match_data['outcome'].value_counts())
print(f"\nSample data:")
print(match_data.head())

## Prepare Data

In [ ]:
# Features and target
feature_cols = ['home_shots', 'away_shots', 'home_passes', 'away_passes', 'home_tackles', 'away_tackles']
X = match_data[feature_cols]
y = match_data['outcome']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")

## Train Random Forest

In [ ]:
# Train Random Forest with 100 trees
rf_clf = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

rf_clf.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_clf.predict(X_test)

# Evaluate
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {accuracy_rf:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

## Compare with Single Decision Tree

In [ ]:
# Train a single decision tree for comparison
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_clf.fit(X_train, y_train)
y_pred_tree = tree_clf.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)

print(f"Single Tree Accuracy: {accuracy_tree:.3f}")
print(f"Random Forest Accuracy: {accuracy_rf:.3f}")
print(f"\nImprovement: {(accuracy_rf - accuracy_tree)*100:.1f} percentage points")

## Feature Importance

Random Forests can tell us which features are most important for predictions.

In [ ]:
# Get feature importances
importances = rf_clf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance', fontsize=12)
plt.title('Random Forest Feature Importance for Match Outcome', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nInterpretation: {feature_importance_df.iloc[0]['Feature']} is the most important feature")

## Understanding Out-of-Bag (OOB) Error

Random Forests have a built-in validation mechanism: **Out-of-Bag (OOB) error**.

Since each tree trains on a bootstrap sample (~63% of data), the remaining ~37% can be used for validation.

In [ ]:
# Train with OOB score enabled
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
rf_oob.fit(X_train, y_train)

print(f"OOB Score: {rf_oob.oob_score_:.3f}")
print(f"Test Accuracy: {rf_oob.score(X_test, y_test):.3f}")
print(f"\nOOB score is a free validation estimate without needing a separate validation set!")

## Tuning Random Forests

Key hyperparameters:
- **n_estimators**: Number of trees (more is generally better)
- **max_depth**: Maximum depth of each tree
- **max_features**: Features to consider at each split
- **min_samples_split**: Minimum samples to split a node

In [ ]:
# Quick tuning experiment: number of trees
n_trees_range = [10, 50, 100, 200, 300]
accuracies = []

for n_trees in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    accuracies.append(rf.score(X_test, y_test))
    print(f"n_estimators={n_trees:3d}: Accuracy = {accuracies[-1]:.3f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(n_trees_range, accuracies, marker='o', linewidth=2)
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Effect of Number of Trees on Performance', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## When to Use Random Forest

### Use Random Forest when:
- ✅ You need a **quick, robust baseline** with minimal tuning
- ✅ **Feature importance** is sufficient for interpretability
- ✅ You have **small to medium datasets**
- ✅ You want **stable, reliable predictions**

### Don't use Random Forest when:
- ❌ You need **maximum accuracy** (use XGBoost instead)
- ❌ You need **full model interpretability** (use single decision tree)
- ❌ Training time is critical (Random Forest can be slow)
- ❌ You have **very high-dimensional sparse data**

## Summary

In this notebook, we:

1. ✅ Understood ensemble learning (bagging + feature randomness)
2. ✅ Built a Random Forest for match outcome prediction
3. ✅ Interpreted feature importances
4. ✅ Compared with single decision trees
5. ✅ Learned about OOB error
6. ✅ Tuned the number of trees

## Key Takeaways

- Random Forest = **many diverse trees voting**
- **More stable and accurate** than single trees
- **Feature importance** reveals what drives predictions
- **OOB score** provides free validation
- Great **default choice** for many classification tasks

## Next Steps

Random Forest is powerful, but **XGBoost** can squeeze out even more accuracy through sequential learning. Let's explore it next!

## Practice Exercises

1. **Grid Search**: Tune max_depth and max_features together
2. **Real Data**: Use actual StatsBomb match data instead of synthetic
3. **Probability Predictions**: Use predict_proba() to get confidence scores
4. **Partial Dependence**: Plot how predictions change with one feature
5. **Compare Algorithms**: Try ExtraTreesClassifier and compare with RandomForest